# Обучение ResNet/EfficientNet для детекции 4 углов шахматной доски

Этот блокнот обучает модель (ResNet18/34/50 или EfficientNet), которая по изображению доски предсказывает 4 угла (8 координат).

🔄 **Data Augmentation:**
- Модель обучается с **случайными отражениями** (горизонтальными и вертикальными)
- Это увеличивает разнообразие данных без риска обрезания точек
- Координаты правильно трансформируются при отражениях
- После каждой трансформации применяется канонический порядок точек (TL, TR, BR, BL)
- ⚠️ Повороты убраны, т.к. они обрезают точки при expand=False

⚠️ **Важно:** блокнот рассчитан на запуск в Google Colab.

Формат датасета в архиве (`.zip`):

- В корне архива должна быть папка `chess-boards-resnet/` со структурой:
  - `chess-boards-resnet/train/images/*.jpg`
  - `chess-boards-resnet/train/labels/*.txt`
  - `chess-boards-resnet/valid/images/*.jpg`
  - `chess-boards-resnet/valid/labels/*.txt`
  - `chess-boards-resnet/test/...` (опционально)

Формат каждого `.txt`-лейбла:
```
x1 y1 x2 y2 x3 y3 x4 y4
```
где `x`, `y` — нормализованы в диапазон `[0, 1]` относительно ширины/высоты исходного изображения.

## 1. Загрузка и распаковка архива с датасетом

In [ ]:
# Подготовка путей к датасету (архив уже лежит в файловой системе)
import os, zipfile
from pathlib import Path

IN_COLAB = 'google.colab' in str(get_ipython())
print('IN_COLAB =', IN_COLAB)

if IN_COLAB:
    # Монтируем Google Drive
    from google.colab import drive
    drive.mount('/content/drive')
    
    # Архив должен быть в корне Google Drive
    ZIP_NAME = 'chess-boards-resnet.zip'  # <--- поменяй на своё имя, если нужно
    zip_path = Path('/content/drive/MyDrive') / ZIP_NAME
    
    if not zip_path.exists():
        raise FileNotFoundError(f'Архив не найден: {zip_path}. Убедитесь, что файл {ZIP_NAME} находится в корне Google Drive.')
else:
    # Локальный запуск: ищем первый .zip в текущей директории
    zips = [f for f in os.listdir('.') if f.lower().endswith('.zip')]
    if not zips:
        raise RuntimeError('Не найден ни один .zip в текущей директории')
    zip_path = Path(zips[0])

print('Используем архив:', zip_path)

# Каталог проекта (куда распаковываем)
PROJECT_DIR = Path('/content/chess_board_corners_resnet') if IN_COLAB else Path.cwd() / 'chess_board_corners_resnet'
PROJECT_DIR.mkdir(parents=True, exist_ok=True)
print('PROJECT_DIR =', PROJECT_DIR)

# Удаляем старую папку chess-boards-resnet, если она существует
old_dataset_dir = PROJECT_DIR / 'chess-boards-resnet'
if old_dataset_dir.exists():
    import shutil
    print(f'Удаляем старый датасет: {old_dataset_dir}')
    shutil.rmtree(old_dataset_dir)

# Распаковка архива
print('Распаковка архива...')
with zipfile.ZipFile(zip_path, 'r') as zf:
    zf.extractall(PROJECT_DIR)
print('Архив распакован.')

# Пытаемся найти корень датасета внутри PROJECT_DIR
candidates = list(PROJECT_DIR.glob('chess-boards-resnet'))
if candidates:
    DATASET_DIR = candidates[0]
else:
    # возможно, архив уже содержал структуру сразу в PROJECT_DIR
    DATASET_DIR = PROJECT_DIR

print('DATASET_DIR =', DATASET_DIR)

# Проверим наличие train/valid
train_images = DATASET_DIR / 'train' / 'images'
val_images = DATASET_DIR / 'valid' / 'images'
print('train images dir exists:', train_images.exists())
print('valid images dir exists:', val_images.exists())


## 2. Импорт PyTorch и описание Dataset/DataLoader

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from torchvision import models, transforms
from PIL import Image
import numpy as np
from tqdm.auto import tqdm

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
IMG_SIZE = 640  # Увеличено для ResNet34 - больше деталей
BATCH_SIZE = 12  # Уменьшено из-за увеличения размера изображения (может потребоваться дальнейшее уменьшение)
EPOCHS = 100  # Увеличено для лучшей сходимости
LR = 1e-4
WEIGHT_DECAY = 1e-4
PATIENCE = 15  # Early stopping patience

# Выбор модели: 'resnet18', 'resnet34', 'resnet50', 'efficientnet_b0', 'efficientnet_b3'
# ResNet34/50 - больше параметров, лучше точность, но медленнее
# EfficientNet - современная архитектура, хороший баланс точности и скорости
MODEL_NAME = 'resnet34'  # Измените на 'resnet50' или 'efficientnet_b3' для лучшей точности

MODELS_DIR = PROJECT_DIR / 'models_resnet'
MODELS_DIR.mkdir(exist_ok=True)
print('DEVICE:', DEVICE)
print('MODELS_DIR:', MODELS_DIR)


In [ ]:
class BoardCornersDataset(Dataset):
    """
    Ожидает структуру:
      images_dir/*.jpg
      labels_dir/*.txt (8 чисел: x1 y1 ... x4 y4 в [0,1])
    """
    def __init__(self, images_dir: Path, labels_dir: Path, img_size: int = 256, augment: bool = False):
        self.images = sorted(list(images_dir.glob('*.jpg')))
        self.labels_dir = labels_dir
        self.img_size = img_size
        self.augment = augment

        # Базовые трансформации (без augmentation)
        self.base_transform = transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                 std=[0.229, 0.224, 0.225]),
        ])

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_path = self.images[idx]
        label_path = self.labels_dir / (img_path.stem + '.txt')

        img = Image.open(img_path).convert('RGB')
        img_w, img_h = img.size

        coords = np.loadtxt(label_path, dtype=np.float32).reshape(-1)
        if coords.shape[0] != 8:
            raise ValueError(f'Ожидалось 8 чисел в {label_path}, найдено {coords.shape[0]}')

        # Data augmentation: только отражения (без поворотов, чтобы точки не выходили за границы)
        if self.augment:
            import random
            
            if random.random() < 0.5:
                # Выбираем случайный масштаб от 50% до 80% (zoom out)
                zoom_factor = random.uniform(0.5, 0.8)
                new_w = int(img_w * zoom_factor)
                new_h = int(img_h * zoom_factor)
                
                # Ресайзим изображение до меньшего размера
                img_resized = img.resize((new_w, new_h), Image.Resampling.LANCZOS)
                
                # Создаем новое изображение исходного размера с черным фоном
                img_zoomed = Image.new('RGB', (img_w, img_h), (0, 0, 0))
                
                # Вычисляем позицию для центрирования уменьшенного изображения
                x_offset = (img_w - new_w) // 2
                y_offset = (img_h - new_h) // 2
                
                # Вставляем уменьшенное изображение в центр
                img_zoomed.paste(img_resized, (x_offset, y_offset))
                img = img_zoomed
                
                # Трансформируем координаты: они масштабируются и смещаются
                # Координаты нормализованы [0,1], поэтому:
                # x_new = (x_old * img_w * zoom_factor + offset) / img_w
                # Упрощаем: x_new = x_old * zoom_factor + offset/img_w
                offset_x_norm = x_offset / img_w
                offset_y_norm = y_offset / img_h
                coords[0::2] = coords[0::2] * zoom_factor + offset_x_norm  # x координаты
                coords[1::2] = coords[1::2] * zoom_factor + offset_y_norm  # y координаты
                
                # Обрезаем координаты в диапазон [0, 1] (на случай если вышли за границы)
                coords = np.clip(coords, 0, 1)
                
                # Применяем канонический порядок после zoom out
                coords = self.apply_canonical_order(coords)
                
                # Случайное горизонтальное отражение (50% вероятность)
                if random.random() < 0.5:
                    img = img.transpose(Image.FLIP_LEFT_RIGHT)
                    # Отражаем координаты: x -> 1 - x
                    coords[0::2] = 1.0 - coords[0::2]
                    # Применяем канонический порядок после отражения
                    coords = self.apply_canonical_order(coords)
            
            # Случайное вертикальное отражение (50% вероятность)
            if random.random() < 0.5:
                img = img.transpose(Image.FLIP_TOP_BOTTOM)
                # Отражаем координаты: y -> 1 - y
                coords[1::2] = 1.0 - coords[1::2]
                # Применяем канонический порядок после отражения
                coords = self.apply_canonical_order(coords)
        
        target = torch.from_numpy(coords).float()
        img_t = self.base_transform(img)
        return img_t, target
    
    def apply_canonical_order(self, coords: np.ndarray) -> np.ndarray:
        """Применяет канонический порядок точек (TL, TR, BR, BL)."""
        points = coords.reshape(4, 2)
        
        # Сортируем по Y (меньший Y = выше)
        idx_by_y = np.argsort(points[:, 1])
        top = points[idx_by_y[:2]]
        bottom = points[idx_by_y[2:]]
        
        # Внутри top/bottom сортируем по X
        top = top[np.argsort(top[:, 0])]
        bottom = bottom[np.argsort(bottom[:, 0])]
        
        # Собираем в порядке TL, TR, BR, BL
        ordered = np.stack([top[0], top[1], bottom[1], bottom[0]], axis=0)
        return ordered.reshape(-1)


# Инициализация датасетов и лоадеров
# ВАЖНО: augmentation только для train, не для validation!
train_ds = BoardCornersDataset(
    DATASET_DIR / 'train' / 'images',
    DATASET_DIR / 'train' / 'labels',
    img_size=IMG_SIZE,
    augment=True,  # Включаем augmentation для обучения
)
val_ds = BoardCornersDataset(
    DATASET_DIR / 'valid' / 'images',
    DATASET_DIR / 'valid' / 'labels',
    img_size=IMG_SIZE,
    augment=False,  # Без augmentation для валидации
)

print('Train samples:', len(train_ds))
print('Val samples:', len(val_ds))

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,      # без отдельных процессов
    pin_memory=False    # можно тоже убрать
)

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=False
)

## 3. Модель ResNet18 → 8 координат

In [ ]:
class CornerRegressor(nn.Module):
    def __init__(self, model_name: str = 'resnet18', pretrained: bool = True):
        super().__init__()
        self.model_name = model_name
        
        # Загружаем backbone в зависимости от выбранной модели
        if model_name.startswith('resnet'):
            if model_name == 'resnet18':
                self.backbone = models.resnet18(
                    weights=models.ResNet18_Weights.IMAGENET1K_V1 if pretrained else None
                )
            elif model_name == 'resnet34':
                self.backbone = models.resnet34(
                    weights=models.ResNet34_Weights.IMAGENET1K_V1 if pretrained else None
                )
            elif model_name == 'resnet50':
                self.backbone = models.resnet50(
                    weights=models.ResNet50_Weights.IMAGENET1K_V1 if pretrained else None
                )
            else:
                raise ValueError(f"Неизвестная ResNet модель: {model_name}")
            
            in_features = self.backbone.fc.in_features
            # Используем два слоя для лучшей регрессии
            self.backbone.fc = nn.Sequential(
                nn.Linear(in_features, 512),
                nn.ReLU(),
                nn.Dropout(0.2),
                nn.Linear(512, 8)
            )
        
        elif model_name.startswith('efficientnet'):
            try:
                from torchvision.models import efficientnet_b0, efficientnet_b3, EfficientNet_B0_Weights, EfficientNet_B3_Weights
            except ImportError:
                # Для старых версий torchvision
                from torchvision.models import efficientnet_b0, efficientnet_b3
                EfficientNet_B0_Weights = None
                EfficientNet_B3_Weights = None
            
            if model_name == 'efficientnet_b0':
                if EfficientNet_B0_Weights:
                    self.backbone = efficientnet_b0(
                        weights=EfficientNet_B0_Weights.IMAGENET1K_V1 if pretrained else None
                    )
                else:
                    self.backbone = efficientnet_b0(pretrained=pretrained)
                in_features = self.backbone.classifier[1].in_features
                self.backbone.classifier = nn.Sequential(
                    nn.Linear(in_features, 512),
                    nn.ReLU(),
                    nn.Dropout(0.2),
                    nn.Linear(512, 8)
                )
            elif model_name == 'efficientnet_b3':
                if EfficientNet_B3_Weights:
                    self.backbone = efficientnet_b3(
                        weights=EfficientNet_B3_Weights.IMAGENET1K_V1 if pretrained else None
                    )
                else:
                    self.backbone = efficientnet_b3(pretrained=pretrained)
                in_features = self.backbone.classifier[1].in_features
                self.backbone.classifier = nn.Sequential(
                    nn.Linear(in_features, 512),
                    nn.ReLU(),
                    nn.Dropout(0.2),
                    nn.Linear(512, 8)
                )
            else:
                raise ValueError(f"Неизвестная EfficientNet модель: {model_name}")
        else:
            raise ValueError(f"Неизвестная модель: {model_name}")

    def forward(self, x):
        output = self.backbone(x)
        # Применяем sigmoid для гарантии диапазона [0,1]
        return torch.sigmoid(output)


print(f"Используем модель: {MODEL_NAME}")
model = CornerRegressor(model_name=MODEL_NAME, pretrained=True).to(DEVICE)

# Подсчитываем количество параметров
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Всего параметров: {total_params:,}")
print(f"Обучаемых параметров: {trainable_params:,}")
# Комбинируем SmoothL1Loss и MSE для лучшей сходимости
criterion = nn.SmoothL1Loss()
mse_criterion = nn.MSELoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
# Learning rate scheduler для адаптивного уменьшения LR
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=5
)

print(model)


## 4. Цикл обучения

In [ ]:
def train_one_epoch(model, loader, optimizer, device):
    model.train()
    running_loss = 0.0
    for imgs, targets in tqdm(loader, desc='Train', leave=False):
        imgs = imgs.to(device)
        targets = targets.to(device)

        optimizer.zero_grad()
        outputs = model(imgs)  # B x 8
        # Комбинируем два loss для лучшей сходимости
        loss = criterion(outputs, targets) + 0.1 * mse_criterion(outputs, targets)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * imgs.size(0)

    return running_loss / len(loader.dataset)


def eval_one_epoch(model, loader, device):
    model.eval()
    running_loss = 0.0
    with torch.no_grad():
        for imgs, targets in tqdm(loader, desc='Val', leave=False):
            imgs = imgs.to(device)
            targets = targets.to(device)

            outputs = model(imgs)
            loss = criterion(outputs, targets) + 0.1 * mse_criterion(outputs, targets)
            running_loss += loss.item() * imgs.size(0)

    return running_loss / len(loader.dataset)


best_val_loss = float('inf')
best_train_loss = float('inf')
best_model_path = MODELS_DIR / f'best_{MODEL_NAME}_board_corners.pt'
patience_counter = 0

for epoch in range(1, EPOCHS + 1):
    print(f"\nEpoch {epoch}/{EPOCHS}")
    train_loss = train_one_epoch(model, train_loader, optimizer, DEVICE)
    val_loss = eval_one_epoch(model, val_loader, DEVICE)

    # Обновляем learning rate
    scheduler.step(val_loss)
    current_lr = optimizer.param_groups[0]['lr']

    print(f"Train loss: {train_loss:.4f}")
    print(f"Val   loss: {val_loss:.4f}")
    print(f"LR: {current_lr:.6f}")

    # Исправленная логика: сохраняем только при реальном улучшении
    val_improved = val_loss < best_val_loss - 1e-6
    train_improved = train_loss < best_train_loss - 1e-6
    
    if val_improved:
        # Val loss улучшился - всегда сохраняем
        best_val_loss = val_loss
        best_train_loss = train_loss
        patience_counter = 0
        torch.save(model.state_dict(), best_model_path)
        print(f">>> New best model saved (val_loss improved to {val_loss:.6f})")
    elif val_loss <= best_val_loss + 1e-6 and train_improved:
        # Val loss не ухудшился (равен или лучше) И train loss улучшился - сохраняем
        best_val_loss = val_loss
        best_train_loss = train_loss
        patience_counter = 0
        torch.save(model.state_dict(), best_model_path)
        print(f">>> New best model saved (train_loss improved to {train_loss:.6f}, val_loss={val_loss:.6f})")
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"\nEarly stopping after {epoch} epochs (patience={PATIENCE})")
            break

## 5. Визуализация предсказанных углов на изображениях

In [ ]:
# Загрузка лучшей модели и визуализация предсказаний на нескольких изображениях
from matplotlib import pyplot as plt

# загрузим лучшую модель
best_model = CornerRegressor(model_name=MODEL_NAME, pretrained=False).to(DEVICE)
best_model.load_state_dict(torch.load(best_model_path, map_location=DEVICE))
best_model.eval()


def denorm_coords(coords, w, h):
    """Преобразовать нормализованные координаты [0,1] → пиксели."""
    xs = coords[0::2] * w
    ys = coords[1::2] * h
    return xs, ys


# возьмём несколько картинок из валидации
val_imgs = list((DATASET_DIR / 'test' / 'images').glob('*.jpg'))[:5]
print('Проверка на', len(val_imgs), 'валидационных изображениях')

for img_path in val_imgs:
    lbl_path = DATASET_DIR / 'test' / 'labels' / (img_path.stem + '.txt')
    if not lbl_path.exists():
        continue

    img = Image.open(img_path).convert('RGB')
    w, h = img.size
    gt = np.loadtxt(lbl_path, dtype=np.float32).reshape(-1)  # 8 чисел

    # подготовим тензор
    x = transforms.Resize((IMG_SIZE, IMG_SIZE))(img)
    x = transforms.ToTensor()(x)
    x = transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225])(x)
    x = x.unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        pred = best_model(x).cpu().numpy().reshape(-1)  # 8 чисел

    gt_xs, gt_ys = denorm_coords(gt, w, h)
    pr_xs, pr_ys = denorm_coords(pred, w, h)

    # Вычисляем среднюю ошибку
    errors = [np.sqrt((gt_xs[i] - pr_xs[i])**2 + (gt_ys[i] - pr_ys[i])**2) for i in range(4)]
    mean_error = np.mean(errors)
    max_error = np.max(errors)

    print(f"\n{img_path.name} (размер: {w}x{h}):")
    print(f"  Средняя ошибка: {mean_error:.1f} px, Максимальная: {max_error:.1f} px")
    for i in range(4):
        print(f"  Точка {i}: GT=({gt_xs[i]:.1f}, {gt_ys[i]:.1f})  "
              f"Pred=({pr_xs[i]:.1f}, {pr_ys[i]:.1f})  err={errors[i]:.1f} px")
    print(f"  GT норм: {gt}")
    print(f"  Pred норм: {pred}")

    # рисуем
    fig, ax = plt.subplots(1, 1, figsize=(10, 12))
    ax.imshow(img)
    ax.set_title(f'{img_path.name}\nСредняя ошибка: {mean_error:.1f}px, Макс: {max_error:.1f}px', fontsize=10)

    # GT — зелёные с соединением линиями
    ax.plot([gt_xs[0], gt_xs[1], gt_xs[2], gt_xs[3], gt_xs[0]], 
            [gt_ys[0], gt_ys[1], gt_ys[2], gt_ys[3], gt_ys[0]], 
            'g-', linewidth=2, alpha=0.5, label='GT polygon')
    ax.scatter(gt_xs, gt_ys, c='lime', s=80, marker='o', edgecolors='darkgreen', linewidths=2, label='GT', zorder=5)
    for i in range(4):
        ax.text(gt_xs[i] + 5, gt_ys[i] - 5, f'G{i}', color='lime', fontsize=10, weight='bold')

    # Pred — красные с соединением линиями
    ax.plot([pr_xs[0], pr_xs[1], pr_xs[2], pr_xs[3], pr_xs[0]], 
            [pr_ys[0], pr_ys[1], pr_ys[2], pr_ys[3], pr_ys[0]], 
            'r-', linewidth=2, alpha=0.5, label='Pred polygon')
    ax.scatter(pr_xs, pr_ys, c='red', s=80, marker='x', linewidths=3, label='Pred', zorder=5)
    for i in range(4):
        ax.text(pr_xs[i] + 5, pr_ys[i] + 10, f'P{i}', color='red', fontsize=10, weight='bold')

    # Соединяем GT и Pred линиями для наглядности
    for i in range(4):
        ax.plot([gt_xs[i], pr_xs[i]], [gt_ys[i], pr_ys[i]], 
                'b--', linewidth=1, alpha=0.3)

    ax.legend(loc='upper right')
    ax.axis('off')
    plt.tight_layout()
    plt.show()